In [ ]:
import os
import time
import pandas as pd
from groq import Groq

# Replace with your key or set it in your environment
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
client = Groq()

# Define models to test
# llama-3.1-8b-instant: Ultra-low TTFT
# llama-3.3-70b-versatile: Higher commentary quality & depth
MODELS = ["openai/gpt-oss-120b"]

# 1. Entertaining Host Persona
HOST_SYSTEM_PROMPT = """You are an energetic, witty live chess broadcast host. 
Keep the broadcast entertaining, build tension, use lively banter, and hype up the audience. 
Keep reactions punchy (under 40 words). Speak directly to the stream viewers."""

# 2. Grandmaster Tactical Analyst Persona
GM_SYSTEM_PROMPT = """You are a Grandmaster technical analyst on a chess stream. 
Explain the strategic rationale, calculate concrete tactical lines, evaluate piece activity, and give the critical continuation. 
Keep it concise, analytical, and authoritative (around 40-70 words)."""

# Simulated game event to commentate
SAMPLE_MOVE = {
    "fen": "r1bqk2r/pp2bppp/2n1p3/3pP3/3P4/5N2/PP3PPP/RNBQ1RK1 w kq - 1 10",
    "move": "16. Bxh7+!?",
    "context": "White sacrifices the bishop against Black's castled king. Black's king is forced to accept or decline the Greek Gift sacrifice."
}

def format_event_prompt(event: dict) -> str:
    return f"Move played: {event['move']}\nPosition Context: {event['context']}\nFEN: {event['fen']}\nProvide your commentary now."

In [27]:
import time
import tiktoken
from groq import Groq

client = Groq()

# Fast local tokenizer for accurate token and speed calculations
encoder = tiktoken.get_encoding("cl100k_base")

def benchmark_commentary_stream(model: str, system_prompt: str, user_prompt: str):
    start_time = time.perf_counter()
    first_token_time = None
    text_chunks = []
    reasoning_chunks = []
    
    # Pure streaming call without stream_options
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=600,
        stream=True
    )

    for chunk in stream:
        if not chunk.choices:
            continue

        delta = chunk.choices[0].delta
        content = getattr(delta, "content", None)
        reasoning = getattr(delta, "reasoning_content", None) or getattr(delta, "reasoning", None)

        # Mark TTFT on the very first incoming token (reasoning or final content)
        if (content or reasoning) and first_token_time is None:
            first_token_time = time.perf_counter()

        if content:
            text_chunks.append(content)
        if reasoning:
            reasoning_chunks.append(reasoning)

    end_time = time.perf_counter()

    full_text = "".join(text_chunks).strip()
    full_reasoning = "".join(reasoning_chunks).strip()

    # Determine display content and calculate token volume
    output_body = full_text if full_text else full_reasoning
    total_tokens = len(encoder.encode(output_body)) if output_body else 0

    # Calculate latency metrics
    ttft_ms = (first_token_time - start_time) * 1000 if first_token_time else 0
    total_time_s = end_time - start_time
    gen_time_s = end_time - first_token_time if first_token_time else total_time_s
    tps = total_tokens / gen_time_s if gen_time_s > 0 and total_tokens > 0 else 0

    return {
        "ttft_ms": round(ttft_ms, 2),
        "total_latency_s": round(total_time_s, 3),
        "tokens_generated": total_tokens,
        "tokens_per_sec": round(tps, 1),
        "text": output_body
    }

In [28]:
TRIALS = 3
results = []
prompt_text = format_event_prompt(SAMPLE_MOVE)

test_scenarios = [
    ("Entertaining Host", HOST_SYSTEM_PROMPT),
    ("GM Analyst", GM_SYSTEM_PROMPT)
]

print("Running commentary latency test across models...\n")

for model in MODELS:
    for role_name, sys_prompt in test_scenarios:
        for trial in range(1, TRIALS + 1):
            metric = benchmark_commentary_stream(model, sys_prompt, prompt_text)
            
            results.append({
                "Model": model,
                "Role": role_name,
                "Trial": trial,
                "TTFT (ms)": metric["ttft_ms"],
                "Total Latency (s)": metric["total_latency_s"],
                "Tokens": metric["tokens_generated"],
                "Speed (T/s)": metric["tokens_per_sec"],
                "Snippet": metric["text"][:60] + "..."
            })
            time.sleep(0.5)  # brief pause to avoid hitting strict burst limits

df_raw = pd.DataFrame(results)
print("Benchmarking complete.")

Running commentary latency test across models...

Benchmarking complete.


In [30]:
df_raw

,Model,Role,Trial,TTFT (ms),Total Latency (s),Tokens,Speed (T/s),Snippet
0,openai/gpt-oss-120b,Entertaining Host,1,635.94,1.356,52,72.3,Whoa! Bxh7+! White drops the bishop like a bom...
1,openai/gpt-oss-120b,Entertaining Host,2,438.11,0.948,138,270.4,🔥 **Whoa!** White just drops the bishop on h7 ...
2,openai/gpt-oss-120b,Entertaining Host,3,469.44,1.030,219,390.7,**Whoa!** White drops the bishop on h7 with a ...
3,openai/gpt-oss-120b,GM Analyst,1,130.25,1.393,80,63.3,**Idea:** The classic “Greek Gift”. 16 Bxh7+ f...
4,openai/gpt-oss-120b,GM Analyst,2,589.69,1.646,182,172.3,**Rationale** – Bxh7+ opens the h‑file and for...
5,openai/gpt-oss-120b,GM Analyst,3,495.34,1.669,21,17.9,**16 Bxh7+! – the classic Greek‑Gift** \n\nIf...


In [29]:
# Summary metrics grouped by Model and Commentary Role
summary_df = df_raw.groupby(["Model", "Role"])[
    ["TTFT (ms)", "Total Latency (s)", "Tokens", "Speed (T/s)"]
].mean().reset_index()

print("=== AVERAGE LATENCY & THROUGHPUT ===")
print(summary_df.to_markdown(index=False))

print("\n=== SAMPLE GENERATIONS ===")
for role in ["Entertaining Host", "GM Analyst"]:
    sample = df_raw[(df_raw["Role"] == role) & (df_raw["Model"] == "openai/gpt-oss-120b")].iloc[0]
    print(f"\n[{role.upper()} SAMPLE]:")
    print(sample["Snippet"])

=== AVERAGE LATENCY & THROUGHPUT ===
| Model               | Role              |   TTFT (ms) |   Total Latency (s) |   Tokens |   Speed (T/s) |
|:--------------------|:------------------|------------:|--------------------:|---------:|--------------:|
| openai/gpt-oss-120b | Entertaining Host |     514.497 |             1.11133 | 136.333  |       244.467 |
| openai/gpt-oss-120b | GM Analyst        |     405.093 |             1.56933 |  94.3333 |        84.5   |

=== SAMPLE GENERATIONS ===

[ENTERTAINING HOST SAMPLE]:
Whoa! Bxh7+! White drops the bishop like a bomb—Black’s cast...

[GM ANALYST SAMPLE]:
**Idea:** The classic “Greek Gift”. 16 Bxh7+ forces the king...
